# SWEFil mixed epoch100 — native test submission on T4

Run this **single code cell** in a Colab **T4 GPU** runtime. It predicts all **180 original test images** with the completed GONG/SWEFil/CHASE model selected at **epoch100** (development PQ **0.410866**).

- Frozen validation recipe: ConvNeXt-Tiny U-Net, limb correction and normalization, 1024-pixel tiles with 25% overlap, FP16 batch1, foreground threshold **0.65**, eight-connected components, minimum area **300**.
- This is inference only: no additional training, data synthesis, YOLO fusion, TTA, or public prediction blending.
- The selected weight is read through the **Drive API** from `verified_checkpoints`, using the best file ID in the completed training metadata. Its exact SHA-256 and checkpoint contents are verified. Use the same Google account for Drive mounting and authentication.
- After interruption, run **this same cell**. The saved source and verified completed image predictions are reused. An unfinished image is repeated. A new runtime may download dependencies and the 128.5 MB weight file again.
- At completion, download **`submission_swefil_epoch100_native.csv`** and submit that CSV to Kaggle. The review ZIP is for analysis. The cell itself does not submit to Kaggle or calculate test PQ.

The output folder is `filament-segmentation-2026/submissions/swefil_epoch100_native_t4_v1`. Existing training results and earlier submissions remain in their own folders. Logs are displayed periodically and saved as `inference.log`.

If the completed training metadata is stale, set `CHECKPOINT_FILE_ID` to the reviewed epoch100 best weight's Drive file ID. The selected model hash remains fixed; a different checkpoint is rejected. A local copy of that exact file can instead be supplied as `CHECKPOINT`.


In [ ]:
# Inference only. Run this same cell again after an interruption or runtime reset.
from pathlib import Path
from google.colab import auth, drive, files
import hashlib, json, os, re, signal, subprocess, sys
BASE = Path("/content/drive/MyDrive/filament-segmentation-2026")
TRAINING_ROOT = BASE / "fixed_split_experiments/swefil_all_mixed_native_t4_v1"
TEST_IMAGES = BASE / "MAGFiLO_1.0_Kaggle_2026/test/test_images"
OUTPUT = BASE / "submissions/swefil_epoch100_native_t4_v1"
CHECKPOINT_FILE_ID = None  # Auto-read the reviewed best file ID from training metadata.
CHECKPOINT = None  # Optional local path to the exact reviewed epoch100 weight file.
CHECKOUT = Path("/content/solar_swefil_test_github")
REPOSITORY = "https://github.com/ikjoo5/solar_filament.git"
drive.mount("/content/drive")
if CHECKPOINT is None:
    auth.authenticate_user()  # Use the account mounted as Drive; checkpoint download is read-only.
OUTPUT = OUTPUT.resolve()
def sha(path):
    with Path(path).open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()
pointer = OUTPUT / "test_bootstrap_state.json"
if pointer.is_file():
    if pointer.is_symlink():
        raise ValueError("Unsafe saved bootstrap pointer.")
    state = json.loads(pointer.read_text())
    if state.get("scope") != "swefil_epoch100_native_t4_v1" or state.get("output_root") != str(OUTPUT):
        raise ValueError("This saved source belongs to another inference run.")
    version = state.get("version", "")
    if not re.fullmatch("[0-9a-f]{16}", version):
        raise ValueError("Invalid saved source version.")
    snapshot = OUTPUT / "code" / version
    manifest_path = snapshot / "source_manifest.json"
    if manifest_path.is_symlink() or not snapshot.resolve().is_relative_to(OUTPUT) or sha(manifest_path) != state["manifest_sha256"]:
        raise ValueError("Saved source manifest changed.")
    manifest = json.loads(manifest_path.read_text())
    canonical = json.dumps(manifest, sort_keys=True, separators=(",", ":"), allow_nan=False).encode()
    if hashlib.sha256(canonical).hexdigest()[:16] != version:
        raise ValueError("Saved source version differs from its manifest.")
    bootstrap = snapshot / "tools/swefil_test_colab.py"
    if bootstrap.is_symlink() or not bootstrap.resolve().is_relative_to(snapshot.resolve()):
        raise ValueError("Unsafe saved launcher path.")
    if sha(bootstrap) != manifest.get("tools/swefil_test_colab.py"):
        raise ValueError("Saved launcher checksum changed.")
else:
    if CHECKOUT.exists():
        origin = subprocess.check_output(["git", "-C", str(CHECKOUT), "remote", "get-url", "origin"], text=True).strip()
        dirty = subprocess.check_output(["git", "-C", str(CHECKOUT), "status", "--porcelain"], text=True).strip()
        if origin != REPOSITORY or dirty:
            raise ValueError("Preserve the existing checkout and choose another CHECKOUT path.")
        subprocess.run(["git", "-C", str(CHECKOUT), "fetch", "--depth", "1", "origin", "main"], check=True)
        subprocess.run(["git", "-C", str(CHECKOUT), "checkout", "--detach", "FETCH_HEAD"], check=True)
    else:
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "main", REPOSITORY, str(CHECKOUT)], check=True)
    bootstrap = CHECKOUT / "tools/swefil_test_colab.py"
    if not bootstrap.is_file():
        raise FileNotFoundError("The selected code version lacks the native test launcher.")
command = [sys.executable, "-u", str(bootstrap), "--training-root", str(TRAINING_ROOT),
           "--test-images", str(TEST_IMAGES), "--output", str(OUTPUT)]
if CHECKPOINT is not None: command += ["--checkpoint", str(CHECKPOINT)]
if CHECKPOINT_FILE_ID is not None: command += ["--checkpoint-file-id", str(CHECKPOINT_FILE_ID)]
# Relay child output through the kernel so Colab displays progress and errors.
process = subprocess.Popen(command, start_new_session=True, stdout=subprocess.PIPE,
                           stderr=subprocess.STDOUT, text=True, encoding="utf-8",
                           errors="replace", bufsize=1)
try:
    for line in process.stdout:
        print(line, end="", flush=True)
    status = process.wait()
    if status:
        raise subprocess.CalledProcessError(status, command)
finally:
    if process.poll() is None:
        try: os.killpg(process.pid, signal.SIGTERM)
        except ProcessLookupError: pass
        try: process.wait(timeout=15)
        except subprocess.TimeoutExpired:
            try: os.killpg(process.pid, signal.SIGKILL)
            except ProcessLookupError: pass
            process.wait()
    process.stdout.close()
result = json.loads((OUTPUT / "latest_result.json").read_text())
if result.get("status") != "complete" or result.get("images_completed") != 180 or result.get("checkpoint_epoch") != 100:
    raise RuntimeError("A complete epoch100 prediction for all 180 images is required.")
if Path(result.get("output_root", "")).resolve() != OUTPUT:
    raise ValueError("Result belongs to another output directory.")
if result.get("checkpoint_sha256") != "2b22ed327e2f64902bb7b9b3984586d1ba893f52ada74f557622cce4dac27e40":
    raise ValueError("Result does not use the reviewed epoch100 weights.")
downloads = []
for key in ("csv", "archive"):
    path = Path(result[key]).resolve()
    if Path(result[key]).is_symlink() or not path.is_relative_to(OUTPUT) or not path.is_file() or sha(path) != result.get(key + "_sha256"):
        raise ValueError("Missing or changed output: " + key)
    downloads.append(path)
for path in downloads:
    print("Download:", path, flush=True)
    files.download(str(path))
